[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [2]:
import torch
import math

In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    B, S, D = q.shape
    B, S, D = q.shape
    pos = torch.arange(S, device=q.device)
    idx = torch.arange(D // 2, device=q.device)
    dims = 10000 ** (2 * idx / D)
    theta = pos[:, None] * dims[None, :]
    print(theta.shape, B, S, D)

    sin = torch.sin(theta)
    cos = torch.cos(theta)

    def apply_rope(x, sin, cos):
        B, S, D = x.shape
        x_0 = x[:, :, :D//2]
        x_1 = x[:, :, D//2:]
        y0 = x_0 * cos[None, :, :] - x_1 * sin[None, :, :]
        y1 = x_0 * sin[None, :, :] + x_1 * cos[None, :, :]


In [14]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

torch.Size([8, 8]) 1 8 16


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('rope')